In [ ]:
from langchain_community.graphs import Neo4jGraph

# Connect to Neo4j
graph = Neo4jGraph(
    url="neo4j+s://f5886a71.databases.neo4j.io",
    username="neo4j",
    password="fFCqS5-Ak3ZLppY0p1e2rrL1ZN_FJwt7oH_McxlCh5o"
)

champions_query = """
MATCH (c:Champion)-[r1]->(n1)
OPTIONAL MATCH (n1)-[r2]->(n2)
RETURN c, r1, n1, r2, n2
"""

print("✓ Connected to Neo4j")

/var/folders/_n/46drt9kd1_76dqk_vj0xk9qm0000gn/T/ipykernel_10008/1426366927.py:4: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


✓ Connected to Neo4j


In [11]:
graph.refresh_schema()
print(graph.schema)

[#DCE1]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.64.110', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): TimeoutError(60, 'Operation timed out')
Unable to retrieve routing information
Transaction failed and will be retried in 1.063156913677555s (Unable to retrieve routing information)
[#DCE2]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('si-f5886a71-8c08.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): TimeoutError(60, 'Operation timed out')
Transaction failed and will be retried in 2.0741309029460546s (Failed to read from defunct connection IPv4Address(('si-f5886a71-8c08.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))))


Node properties:
Champion {id: INTEGER, name: STRING, title: STRING, damageType: STRING, attackType: STRING, difficulty: INTEGER}
Role {name: STRING}
Ability {id: STRING, name: STRING, slot: STRING, description: STRING}
AttackType {name: STRING}
CrowdControl {name: STRING}
DamageType {name: STRING}
EffectType {name: STRING}
Stat {name: STRING}
Relationship properties:

The relationships:
(:Champion)-[:HAS_ROLE]->(:Role)
(:Champion)-[:HAS_ABILITY]->(:Ability)
(:Champion)-[:ATTACKS_WITH]->(:AttackType)
(:Ability)-[:APPLIES]->(:CrowdControl)
(:Ability)-[:DEALS]->(:DamageType)
(:Ability)-[:SCALES_WITH]->(:Stat)
(:Ability)-[:HAS_EFFECT]->(:EffectType)


In [31]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import GraphCypherQAChain

llm = ChatOpenAI(model_name="gpt-4", temperature=0)
chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, verbose=True, allow_dangerous_requests=True)

In [ ]:
response = chain.invoke({"query": "list Champions that have both roles Mage and Tank"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Champion)-[:HAS_ROLE]->(r:Role) WHERE r.name IN ['Mage', 'Tank'] 
WITH c, COUNT(r) AS rolesCount 
WHERE rolesCount = 2 
RETURN c.name


[#DE8D]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.64.110', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): TimeoutError(60, 'Operation timed out')
Unable to retrieve routing information
Transaction failed and will be retried in 0.983943378302075s (Unable to retrieve routing information)
[#DE36]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('si-f5886a71-8c08.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): TimeoutError(60, 'Operation timed out')
Transaction failed and will be retried in 1.836071056362587s (Failed to read from defunct connection IPv4Address(('si-f5886a71-8c08.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))))


Full Context:
[{'c.name': 'Galio'}, {'c.name': 'Nunu & Willump'}, {'c.name': 'Singed'}, {'c.name': "Cho'Gath"}, {'c.name': 'Malphite'}]

> Finished chain.


{'query': 'list Champions that have both role Mage and Tank',
 'result': "Galio, Nunu & Willump, Singed, Cho'Gath, and Malphite are champions that have both Mage and Tank roles."}